In [0]:
import requests
import json
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window  import Window

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS cricket")
spark.sql("USE CATALOG cricket")
spark.sql("CREATE SCHEMA IF NOT EXISTS cricket.bronze")

spark.sql("CREATE VOLUME IF NOT EXISTS cricket.bronze.cricket_api_project")

base_path = '/Volumes/cricket/bronze/cricket_api_project'


In [0]:
API_KEY='56bac19c-ced0-4106-840a-37fc36ee209a'


api_url=f"https://api.cricapi.com/v1/currentMatches?apikey={API_KEY}&offset=0"


response = requests.get(api_url)

response.raise_for_status()

api_data = response.json()

print(api_data.keys())

print(json.dumps(api_data,indent= 2)[:2000])



In [0]:
raw_file_path = f'{base_path}/current_matches.json'


with open(raw_file_path,'w') as file:
    json.dump(api_data,file)


print("RAW FILE SAVED:",raw_file_path)

In [0]:
bronze_data=[{
    "source_api":api_url,
    "raw_data":json.dumps(api_data),
    "ingestion_time":None
}]



bronze_schema = StructType([
    StructField("source_api",StringType(),True),
    StructField("raw_data",StringType(),True),
    StructField("ingestion_time",TimestampType(),True)
    ])

bronze_schema



In [0]:
bronze_df = spark.createDataFrame(bronze_data,schema=bronze_schema)\
    .withColumn("ingestion_time",current_timestamp())


bronze_df.write.mode("overwrite").saveAsTable("cricket.bronze.bronze_current_matches")

print("BRONZE TABLE SAVED")




hi
